In [58]:
from google.colab import drive
drive.mount('/content/drive/ded')

Drive already mounted at /content/drive/ded; to attempt to forcibly remount, call drive.mount("/content/drive/ded", force_remount=True).


## TensorFlow/Keras Implementation

### 1. Setup and Imports (TensorFlow/Keras)

In [59]:
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, classification_report
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
import time
import pandas as pd
print(f"TensorFlow Version: {tf.__version__}")


physical_devices = tf.config.list_physical_devices('GPU')
if physical_devices:
    tf.config.experimental.set_memory_growth(physical_devices[0], True)
    print("GPU is available.")
else:
    print("GPU is not available, using CPU.")

TensorFlow Version: 2.20.0
GPU is not available, using CPU.


### Create Plot Directory and Helper Function (TensorFlow/Keras)

In [60]:
plot_dir_tf = '/content/drive/MyDrive/output_folder'
os.makedirs(plot_dir_tf, exist_ok=True)
print(f"TensorFlow Plot directory created at: {plot_dir_tf}")

def save_plot_tf(fig, filename, dpi=600):
    filepath = os.path.join(plot_dir_tf, filename)
    fig.savefig(filepath, dpi=dpi, bbox_inches='tight', format='pdf')
    plt.close(fig)
    print(f"Plot saved to {filepath}")

TensorFlow Plot directory created at: /content/drive/MyDrive/output_folder


### 2. Task 1: Dataset Exploration (TensorFlow/Keras)

#### Load Fashion-MNIST Dataset and Print Dimensions (TensorFlow/Keras)

In [61]:
(train_images_tf, train_labels_tf), (test_images_tf, test_labels_tf) = keras.datasets.fashion_mnist.load_data()

print(f"Training dataset size: {len(train_images_tf)}")
print(f"Testing dataset size: {len(test_images_tf)}")
print(f"Training images shape: {train_images_tf.shape}")
print(f"Training labels shape: {train_labels_tf.shape}")
print(f"Testing images shape: {test_images_tf.shape}")
print(f"Testing labels shape: {test_labels_tf.shape}")

class_names_tf = [
    'T-shirt/top', 'Trouser', 'Pullover', 'Dress', 'Coat',
    'Sandal', 'Shirt', 'Sneaker', 'Bag', 'Ankle boot'
]
print(f"Class names: {class_names_tf}")

Training dataset size: 60000
Testing dataset size: 10000
Training images shape: (60000, 28, 28)
Training labels shape: (60000,)
Testing images shape: (10000, 28, 28)
Testing labels shape: (10000,)
Class names: ['T-shirt/top', 'Trouser', 'Pullover', 'Dress', 'Coat', 'Sandal', 'Shirt', 'Sneaker', 'Bag', 'Ankle boot']


#### Display Ten Sample Fashion-MNIST Images (TensorFlow/Keras)

In [62]:
fig_tf_samples = plt.figure(figsize=(10, 10))
for i in range(10):
    plt.subplot(5, 5, i + 1)
    plt.xticks([])
    plt.yticks([])
    plt.grid(False)
    plt.imshow(train_images_tf[i], cmap=plt.cm.binary)
    plt.xlabel(class_names_tf[train_labels_tf[i]])
plt.suptitle('10 Sample Fashion-MNIST Images (TensorFlow/Keras)', fontsize=16)
plt.tight_layout(rect=[0, 0.03, 1, 0.95])
save_plot_tf(fig_tf_samples, 'TF_Sample_Fashion_MNIST_Images.pdf')
plt.show()

print("Inference: This plot displays a selection of 10 images from the Fashion-MNIST dataset, showing the variety of apparel items and their corresponding class labels. This helps in understanding the visual characteristics of the dataset.")

Plot saved to /content/drive/MyDrive/output_folder/TF_Sample_Fashion_MNIST_Images.pdf
Inference: This plot displays a selection of 10 images from the Fashion-MNIST dataset, showing the variety of apparel items and their corresponding class labels. This helps in understanding the visual characteristics of the dataset.


#### Plot Class Distribution (TensorFlow/Keras)

In [63]:
unique_labels_tf, counts_tf = np.unique(train_labels_tf, return_counts=True)

fig_tf_dist = plt.figure(figsize=(10, 6))
plt.bar(class_names_tf, counts_tf, color='skyblue')
plt.xlabel('Class')
plt.ylabel('Number of Images')
plt.title('Fashion-MNIST Training Data Class Distribution (TensorFlow/Keras)')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
save_plot_tf(fig_tf_dist, 'TF_Fashion_MNIST_Class_Distribution.pdf')
plt.show()

print("Class distribution in training data (TensorFlow/Keras):")
for i, count in enumerate(counts_tf):
    print(f"{class_names_tf[i]}: {count} images")

print("\nInference: This bar plot illustrates that the Fashion-MNIST training dataset has a balanced distribution across all 10 classes, with approximately 6,000 images per class. This equal representation is beneficial for training unbiased classification models.")

Plot saved to /content/drive/MyDrive/output_folder/TF_Fashion_MNIST_Class_Distribution.pdf
Class distribution in training data (TensorFlow/Keras):
T-shirt/top: 6000 images
Trouser: 6000 images
Pullover: 6000 images
Dress: 6000 images
Coat: 6000 images
Sandal: 6000 images
Shirt: 6000 images
Sneaker: 6000 images
Bag: 6000 images
Ankle boot: 6000 images

Inference: This bar plot illustrates that the Fashion-MNIST training dataset has a balanced distribution across all 10 classes, with approximately 6,000 images per class. This equal representation is beneficial for training unbiased classification models.


### 3. Task 2: Data Preprocessing (TensorFlow/Keras)

#### Flatten Images, Normalize Pixels, and One-Hot Encode Labels (TensorFlow/Keras)

In [64]:
print("Shapes before preprocessing (TensorFlow/Keras):")
print(f"Training images shape: {train_images_tf.shape}")
print(f"Training labels shape: {train_labels_tf.shape}")
print(f"Testing images shape: {test_images_tf.shape}")
print(f"Testing labels shape: {test_labels_tf.shape}")

train_images_normalized_tf = train_images_tf.astype('float32') / 255.0
test_images_normalized_tf = test_images_tf.astype('float32') / 255.0

train_images_flat_tf = train_images_normalized_tf.reshape(train_images_normalized_tf.shape[0], -1)
test_images_flat_tf = test_images_normalized_tf.reshape(test_images_normalized_tf.shape[0], -1)

num_classes_tf = len(class_names_tf)
train_labels_one_hot_tf = keras.utils.to_categorical(train_labels_tf, num_classes=num_classes_tf)
test_labels_one_hot_tf = keras.utils.to_categorical(test_labels_tf, num_classes=num_classes_tf)

print("\nShapes after preprocessing (TensorFlow/Keras):")
print(f"Flattened and normalized training images shape: {train_images_flat_tf.shape}")
print(f"Flattened and normalized testing images shape: {test_images_flat_tf.shape}")
print(f"One-hot encoded training labels shape: {train_labels_one_hot_tf.shape}")
print(f"One-hot encoded testing labels shape: {test_labels_one_hot_tf.shape}")

Shapes before preprocessing (TensorFlow/Keras):
Training images shape: (60000, 28, 28)
Training labels shape: (60000,)
Testing images shape: (10000, 28, 28)
Testing labels shape: (10000,)

Shapes after preprocessing (TensorFlow/Keras):
Flattened and normalized training images shape: (60000, 784)
Flattened and normalized testing images shape: (10000, 784)
One-hot encoded training labels shape: (60000, 10)
One-hot encoded testing labels shape: (10000, 10)


### 4. Task 3: Model Construction (TensorFlow/Keras)

#### Construct Baseline MLP Model (TensorFlow/Keras)

In [65]:

input_size_tf = train_images_flat_tf.shape[1]
num_classes_tf_model = num_classes_tf


hidden_size1_tf = 128
hidden_size2_tf = 64


model_tf = keras.Sequential([
    keras.Input(shape=(input_size_tf,)),
    layers.Dense(hidden_size1_tf, activation='relu', name='fc1'),
    layers.Dense(hidden_size2_tf, activation='relu', name='fc2'),
    layers.Dense(num_classes_tf_model, activation='softmax', name='output_layer')
])

model_tf.summary()

print("Inference: This summary provides a detailed overview of the TensorFlow/Keras baseline MLP model, including the output shape of each layer and the total number of trainable parameters. It confirms the successful creation of the network architecture matching the PyTorch version.")

Model: "sequential_2"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ fc1 (Dense)                     │ (None, 128)            │       100,480 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ fc2 (Dense)                     │ (None, 64)             │         8,256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ output_layer (Dense)            │ (None, 10)             │           650 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 109,386 (427.29 KB)

 Trainable params: 109,386 (427.29 KB)

 Non-trainable params: 0 (0.00 B)

Inference: This summary provides a detailed overview of the TensorFlow/Keras baseline MLP model, including the output shape of each layer and the total number of trainable parameters. It confirms the successful creation of the network architecture matching the PyTorch version.


### 5. Task 4: Model Training (TensorFlow/Keras)

#### Compile and Train the Model (TensorFlow/Keras)

In [ ]:

learning_rate_tf = 0.001
batch_size_tf = 32
epochs_tf = 20


model_tf.compile(
    optimizer=keras.optimizers.Adam(learning_rate=learning_rate_tf),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

print(f"Training model for {epochs_tf} epochs with batch size {batch_size_tf} and learning rate {learning_rate_tf}")


start_time_tf = time.time()
history_tf = model_tf.fit(
    train_images_flat_tf,
    train_labels_one_hot_tf,
    batch_size=batch_size_tf,
    epochs=epochs_tf,
    validation_split=0.2,
    verbose=1
)
end_time_tf = time.time()
training_time_tf = end_time_tf - start_time_tf

print(f"\nTraining complete in {training_time_tf:.2f} seconds")

print("Inference: The model has been compiled and trained on the Fashion-MNIST dataset. The training history, including loss and accuracy for both training and validation sets over each epoch, is stored. This step prepares the model for evaluation and provides insights into its learning progress.")

Training model for 20 epochs with batch size 32 and learning rate 0.001
Epoch 1/20
 103/1500 ━━━━━━━━━━━━━━━━━━━━ 7s 6ms/step - accuracy: 0.5341 - loss: 1.3237

In [33]:
fig_baseline_tf_acc = plt.figure(figsize=(10, 6))
plt.plot(history_tf.history['accuracy'], label='Training Accuracy')
plt.plot(history_tf.history['val_accuracy'], label='Validation Accuracy')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.title('Baseline Model Training & Validation Accuracy vs. Epoch (TensorFlow/Keras)')
plt.legend()
plt.grid(True)
save_plot_tf(fig_baseline_tf_acc, 'TF_Baseline_Model_Accuracy_vs_Epoch.pdf')
plt.show()

print("Inference: This plot illustrates the baseline model's training and validation accuracy across epochs, providing insights into its learning trajectory and potential for overfitting or underfitting before hyperparameter tuning.")

Plot saved to /content/drive/MyDrive/output_folder/TF_Baseline_Model_Accuracy_vs_Epoch.pdf
Inference: This plot illustrates the baseline model's training and validation accuracy across epochs, providing insights into its learning trajectory and potential for overfitting or underfitting before hyperparameter tuning.


In [34]:
fig_baseline_tf_loss = plt.figure(figsize=(10, 6))
plt.plot(history_tf.history['loss'], label='Training Loss')
plt.plot(history_tf.history['val_loss'], label='Validation Loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.title('Baseline Model Training & Validation Loss vs. Epoch (TensorFlow/Keras)')
plt.legend()
plt.grid(True)
save_plot_tf(fig_baseline_tf_loss, 'TF_Baseline_Model_Loss_vs_Epoch.pdf')
plt.show()

print("Inference: This plot displays the baseline model's training and validation loss over epochs. It helps in understanding the model's convergence behavior and stability during the initial training phase.")

Plot saved to /content/drive/MyDrive/output_folder/TF_Baseline_Model_Loss_vs_Epoch.pdf
Inference: This plot displays the baseline model's training and validation loss over epochs. It helps in understanding the model's convergence behavior and stability during the initial training phase.


### 6. Task 5: Model Evaluation (TensorFlow/Keras)

#### Compute Metrics and Display Confusion Matrix (TensorFlow/Keras)

In [28]:
optimized_loss_tf, accuracy_tf = model_tf.evaluate(test_images_flat_tf, test_labels_one_hot_tf, verbose=0)
print(f"\nTest Loss (TensorFlow/Keras): {optimized_loss_tf:.4f}")
print(f"Test Accuracy (TensorFlow/Keras): {accuracy_tf:.4f}")

predictions_tf = model_tf.predict(test_images_flat_tf)
predicted_labels_tf = np.argmax(predictions_tf, axis=1)
true_labels_tf = np.argmax(test_labels_one_hot_tf, axis=1)

precision_tf = precision_score(true_labels_tf, predicted_labels_tf, average='weighted', zero_division=0)
recall_tf = recall_score(true_labels_tf, predicted_labels_tf, average='weighted', zero_division=0)
f1_tf = f1_score(true_labels_tf, predicted_labels_tf, average='weighted', zero_division=0)

print(f"Test Precision (TensorFlow/Keras): {precision_tf:.4f}")
print(f"Test Recall (TensorFlow/Keras): {recall_tf:.4f}")
print(f"Test F1-score (TensorFlow/Keras): {f1_tf:.4f}")

print("\nClassification Report (TensorFlow/Keras):\n")
print(classification_report(true_labels_tf, predicted_labels_tf, target_names=class_names_tf, zero_division=0))

cm_tf = confusion_matrix(true_labels_tf, predicted_labels_tf)

fig_tf_cm = plt.figure(figsize=(10, 8))
sns.heatmap(cm_tf, annot=True, fmt='d', cmap='Blues', xticklabels=class_names_tf, yticklabels=class_names_tf)
plt.xlabel('Predicted Label')
plt.ylabel('True Label')
plt.title('Confusion Matrix - Baseline Model (TensorFlow/Keras)')
save_plot_tf(fig_tf_cm, 'TF_Baseline_Confusion_Matrix.pdf')
plt.show()

print("Inference: This confusion matrix for the TensorFlow/Keras baseline model provides a visual breakdown of classification performance across classes. It helps in identifying patterns of correct and incorrect predictions, similar to its PyTorch counterpart.")


Test Loss (TensorFlow/Keras): 0.4799
Test Accuracy (TensorFlow/Keras): 0.8891
313/313 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step
Test Precision (TensorFlow/Keras): 0.8897
Test Recall (TensorFlow/Keras): 0.8891
Test F1-score (TensorFlow/Keras): 0.8891

Classification Report (TensorFlow/Keras):

              precision    recall  f1-score   support

 T-shirt/top       0.84      0.81      0.83      1000
     Trouser       0.99      0.97      0.98      1000
    Pullover       0.84      0.76      0.80      1000
       Dress       0.88      0.91      0.89      1000
        Coat       0.79      0.84      0.82      1000
      Sandal       0.98      0.97      0.97      1000
       Shirt       0.70      0.72      0.71      1000
     Sneaker       0.95      0.96      0.96      1000
         Bag       0.96      0.98      0.97      1000
  Ankle boot       0.96      0.97      0.96      1000

    accuracy                           0.89     10000
   macro avg       0.89      0.89      0.89     10000
weighted a

### 7. Hyperparameter Optimization (TensorFlow/Keras)

In [29]:
!pip install keras-tuner
import keras_tuner as kt

#### Flexible MLP Model for Hyperparameter Tuning (TensorFlow/Keras)

In [46]:
def build_flexible_mlp_tf(hp):
    model = keras.Sequential()
    model.add(keras.Input(shape=(input_size_tf,)))

    num_hidden_layers = hp.Int('num_hidden_layers', min_value=1, max_value=3, default=2)

    for i in range(num_hidden_layers):
        model.add(layers.Dense(units=hp.Int(f'units_{i}', min_value=32, max_value=256, step=32),
                               activation=hp.Choice(f'activation_{i}', values=['relu', 'tanh', 'sigmoid'])))
        dropout_enabled = hp.Boolean(f'dropout_{i}_enabled', default=False)
        if dropout_enabled:
            model.add(layers.Dropout(hp.Float(f'dropout_{i}_rate', min_value=0.0, max_value=0.5, step=0.1, default=0.0)))

    model.add(layers.Dense(num_classes_tf_model, activation='softmax'))

    learning_rate_hp = hp.Choice('learning_rate', values=[1e-2, 1e-3, 1e-4])

    optimizer_choice = hp.Choice('optimizer', values=['adam', 'sgd', 'rmsprop'])
    if optimizer_choice == 'adam':
        optimizer_hp = keras.optimizers.Adam(learning_rate=learning_rate_hp)
    elif optimizer_choice == 'sgd':
        optimizer_hp = keras.optimizers.SGD(learning_rate=learning_rate_hp)
    else:
        optimizer_hp = keras.optimizers.RMSprop(learning_rate=learning_rate_hp)

    model.compile(optimizer=optimizer_hp,
                  loss='categorical_crossentropy',
                  metrics=['accuracy'])

    batch_size_hparam = hp.Choice('batch_size', values=[16, 32, 64, 128], default=32)

    return model

print("Flexible MLP model builder function defined for Keras Tuner.")

Flexible MLP model builder function defined for Keras Tuner.


#### Define the Hyperparameter Search Space (TensorFlow/Keras)

In [47]:
tuner = kt.RandomSearch(
    build_flexible_mlp_tf,
    objective='val_accuracy',
    max_trials=10,
    executions_per_trial=1,
    directory='keras_tuner_dir',
    project_name='fashion_mnist_mlp_tf',
    overwrite=True
)

tuner.search_space_summary()

print("Keras Tuner RandomSearch initialized with objective 'val_accuracy' and a defined search space.")

Search space summary
Default search space size: 10
num_hidden_layers (Int)
{'default': 2, 'conditions': [], 'min_value': 1, 'max_value': 3, 'step': 1, 'sampling': 'linear'}
units_0 (Int)
{'default': None, 'conditions': [], 'min_value': 32, 'max_value': 256, 'step': 32, 'sampling': 'linear'}
activation_0 (Choice)
{'default': 'relu', 'conditions': [], 'values': ['relu', 'tanh', 'sigmoid'], 'ordered': False}
dropout_0_enabled (Boolean)
{'default': False, 'conditions': []}
units_1 (Int)
{'default': None, 'conditions': [], 'min_value': 32, 'max_value': 256, 'step': 32, 'sampling': 'linear'}
activation_1 (Choice)
{'default': 'relu', 'conditions': [], 'values': ['relu', 'tanh', 'sigmoid'], 'ordered': False}
dropout_1_enabled (Boolean)
{'default': False, 'conditions': []}
learning_rate (Choice)
{'default': 0.01, 'conditions': [], 'values': [0.01, 0.001, 0.0001], 'ordered': True}
optimizer (Choice)
{'default': 'adam', 'conditions': [], 'values': ['adam', 'sgd', 'rmsprop'], 'ordered': False}
bat

#### Perform Randomized Search with 5-fold Cross-Validation (TensorFlow/Keras)

In [48]:
print("Starting Randomized Search... (This may take a while)")
search_start_time_tf = time.time()
tuner.search(
    train_images_flat_tf,
    train_labels_one_hot_tf,
    epochs=10,
    validation_split=0.2,
    verbose=0
)
search_end_time_tf = time.time()

print("Randomized Search completed.")

best_hps_tf = tuner.get_best_hyperparameters(num_trials=1)[0]
best_model_tf = tuner.get_best_models(num_models=1)[0]


loss_best_tf, accuracy_best_tf = best_model_tf.evaluate(test_images_flat_tf, test_labels_one_hot_tf, verbose=0)

print(f"\nBest hyperparameters found (TensorFlow/Keras):")
print(f"  Number of Hidden Layers: {best_hps_tf.get('num_hidden_layers')}")
for i in range(best_hps_tf.get('num_hidden_layers')):
    print(f"  Hidden Neurons Layer {i+1}: {best_hps_tf.get(f'units_{i}')}")
    print(f"  Activation Layer {i+1}: {best_hps_tf.get(f'activation_{i}')}")
    dropout_enabled = best_hps_tf.get(f'dropout_{i}_enabled')
    if dropout_enabled:
        print(f"  Dropout Rate Layer {i+1}: {best_hps_tf.get(f'dropout_{i}_rate')}")

print(f"  Learning Rate: {best_hps_tf.get('learning_rate')}")
print(f"  Optimizer: {best_hps_tf.get('optimizer')}")
print(f"  Batch Size: {best_hps_tf.get('batch_size')}")

print(f"Best model test accuracy (from tuner evaluation): {accuracy_best_tf:.4f}")
print(f"Randomized search took {search_end_time_tf - search_start_time_tf:.2f} seconds.")

print("Inference: The Keras Tuner has identified the best set of hyperparameters for the TensorFlow/Keras MLP model based on the specified search space and objective. The best model's test accuracy is now known, which will be used for comparison with the baseline and optimized PyTorch model.")

Starting Randomized Search... (This may take a while)
Randomized Search completed.


/usr/local/lib/python3.12/dist-packages/keras/src/saving/saving_lib.py:797: UserWarning: Skipping variable loading for optimizer 'adam', because it has 2 variables whereas the saved optimizer has 10 variables. 
  saveable.load_own_variables(weights_store.get(inner_path))



Best hyperparameters found (TensorFlow/Keras):
  Number of Hidden Layers: 1
  Hidden Neurons Layer 1: 256
  Activation Layer 1: relu
  Learning Rate: 0.001
  Optimizer: adam
  Batch Size: 64
Best model test accuracy (from tuner evaluation): 0.8770
Randomized search took 843.87 seconds.
Inference: The Keras Tuner has identified the best set of hyperparameters for the TensorFlow/Keras MLP model based on the specified search space and objective. The best model's test accuracy is now known, which will be used for comparison with the baseline and optimized PyTorch model.


### 8. Optimized Model Training and Evaluation (TensorFlow/Keras)

#### 8.1. Train Optimized Model (TensorFlow/Keras)

In [49]:
print("Training optimized model with the best hyperparameters from Keras Tuner...")


optimized_model_tf = tuner.hypermodel.build(best_hps_tf)


epochs_optimized_tf = 20
batch_size_optimized_tf = best_hps_tf.get('batch_size')
print(f"Starting training of optimized model for {epochs_optimized_tf} epochs...")
optimized_train_start_time_tf = time.time()
history_optimized_tf = optimized_model_tf.fit(
    train_images_flat_tf,
    train_labels_one_hot_tf,
    epochs=epochs_optimized_tf,
    batch_size=batch_size_optimized_tf or 32,
    validation_split=0.2,
    verbose=1
)
optimized_train_end_time_tf = time.time()
optimized_training_time_tf = optimized_train_end_time_tf - optimized_train_start_time_tf
print(f"Optimized model training completed in {optimized_training_time_tf:.2f} seconds.")

print("Inference: The optimized TensorFlow/Keras model has been trained using the best hyperparameters found by Keras Tuner. The training history has been captured, allowing for further analysis of its learning process.")

Training optimized model with the best hyperparameters from Keras Tuner...
Starting training of optimized model for 20 epochs...
Epoch 1/20
750/750 ━━━━━━━━━━━━━━━━━━━━ 7s 8ms/step - accuracy: 0.8182 - loss: 0.5216 - val_accuracy: 0.8570 - val_loss: 0.4119
Epoch 2/20
750/750 ━━━━━━━━━━━━━━━━━━━━ 8s 11ms/step - accuracy: 0.8626 - loss: 0.3837 - val_accuracy: 0.8723 - val_loss: 0.3617
Epoch 3/20
750/750 ━━━━━━━━━━━━━━━━━━━━ 5s 7ms/step - accuracy: 0.8753 - loss: 0.3434 - val_accuracy: 0.8658 - val_loss: 0.3602
Epoch 4/20
750/750 ━━━━━━━━━━━━━━━━━━━━ 6s 9ms/step - accuracy: 0.8809 - loss: 0.3225 - val_accuracy: 0.8756 - val_loss: 0.3401
Epoch 5/20
750/750 ━━━━━━━━━━━━━━━━━━━━ 7s 9ms/step - accuracy: 0.8912 - loss: 0.2959 - val_accuracy: 0.8779 - val_loss: 0.3395
Epoch 6/20
750/750 ━━━━━━━━━━━━━━━━━━━━ 5s 7ms/step - accuracy: 0.8961 - loss: 0.2809 - val_accuracy: 0.8817 - val_loss: 0.3302
Epoch 7/20
750/750 ━━━━━━━━━━━━━━━━━━━━ 7s 10ms/step - accuracy: 0.9002 - loss: 0.2678 - val_accuracy:

#### 8.2. Evaluate Optimized Model (TensorFlow/Keras)

In [50]:
optimized_loss_tf, optimized_accuracy_tf = optimized_model_tf.evaluate(test_images_flat_tf, test_labels_one_hot_tf, verbose=0)
print(f"\nOptimized Model Test Loss (TensorFlow/Keras): {optimized_loss_tf:.4f}")
print(f"Optimized Model Test Accuracy (TensorFlow/Keras): {optimized_accuracy_tf:.4f}")

optimized_predictions_tf = optimized_model_tf.predict(test_images_flat_tf)
optimized_predicted_labels_tf = np.argmax(optimized_predictions_tf, axis=1);

optimized_precision_tf = precision_score(true_labels_tf, optimized_predicted_labels_tf, average='weighted', zero_division=0)
optimized_recall_tf = recall_score(true_labels_tf, optimized_predicted_labels_tf, average='weighted', zero_division=0)
optimized_f1_tf = f1_score(true_labels_tf, optimized_predicted_labels_tf, average='weighted', zero_division=0)

print(f"Optimized Model Test Precision (TensorFlow/Keras): {optimized_precision_tf:.4f}")
print(f"Optimized Model Test Recall (TensorFlow/Keras): {optimized_recall_tf:.4f}")
print(f"Optimized Model Test F1-score (TensorFlow/Keras): {optimized_f1_tf:.4f}")

print("\nOptimized Model Classification Report (TensorFlow/Keras):\n")
print(classification_report(true_labels_tf, optimized_predicted_labels_tf, target_names=class_names_tf, zero_division=0))

optimized_cm_tf = confusion_matrix(true_labels_tf, optimized_predicted_labels_tf)

fig_opt_tf_cm = plt.figure(figsize=(10, 8))
sns.heatmap(optimized_cm_tf, annot=True, fmt='d', cmap='Blues', xticklabels=class_names_tf, yticklabels=class_names_tf)
plt.xlabel('Predicted Label')
plt.ylabel('True Label')
plt.title('Confusion Matrix - Optimized Model (TensorFlow/Keras)')
save_plot_tf(fig_opt_tf_cm, 'TF_Optimized_Confusion_Matrix.pdf')
plt.show()

print("Inference: This confusion matrix for the optimized TensorFlow/Keras model visually represents its performance, highlighting correct and incorrect classifications after hyperparameter tuning. It allows for a direct comparison with the baseline and optimized PyTorch models.")


Optimized Model Test Loss (TensorFlow/Keras): 0.3697
Optimized Model Test Accuracy (TensorFlow/Keras): 0.8779
313/313 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step
Optimized Model Test Precision (TensorFlow/Keras): 0.8813
Optimized Model Test Recall (TensorFlow/Keras): 0.8779
Optimized Model Test F1-score (TensorFlow/Keras): 0.8789

Optimized Model Classification Report (TensorFlow/Keras):

              precision    recall  f1-score   support

 T-shirt/top       0.87      0.80      0.83      1000
     Trouser       0.98      0.98      0.98      1000
    Pullover       0.79      0.74      0.77      1000
       Dress       0.91      0.86      0.89      1000
        Coat       0.78      0.82      0.80      1000
      Sandal       0.96      0.97      0.97      1000
       Shirt       0.65      0.75      0.70      1000
     Sneaker       0.96      0.93      0.94      1000
         Bag       0.97      0.97      0.97      1000
  Ankle boot       0.94      0.96      0.95      1000

    accuracy          

#### 8.3. Plots for Optimized Model Training History (TensorFlow/Keras)

In [51]:
fig_opt_tf_acc = plt.figure(figsize=(10, 6))
plt.plot(history_optimized_tf.history['accuracy'], label='Training Accuracy')
plt.plot(history_optimized_tf.history['val_accuracy'], label='Validation Accuracy')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.title('Optimized Model Training & Validation Accuracy vs. Epoch (TensorFlow/Keras)')
plt.legend()
plt.grid(True)
save_plot_tf(fig_opt_tf_acc, 'TF_Optimized_Model_Accuracy_vs_Epoch.pdf')
plt.show()

print("Inference: This plot displays the training and validation accuracy of the optimized TensorFlow/Keras model over epochs. It helps in assessing the model's learning progress and detecting potential overfitting or underfitting after hyperparameter optimization.")


fig_opt_tf_loss = plt.figure(figsize=(10, 6))
plt.plot(history_optimized_tf.history['loss'], label='Training Loss')
plt.plot(history_optimized_tf.history['val_loss'], label='Validation Loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.title('Optimized Model Training & Validation Loss vs. Epoch (TensorFlow/Keras)')
plt.legend()
plt.grid(True)
save_plot_tf(fig_opt_tf_loss, 'TF_Optimized_Model_Loss_vs_Epoch.pdf')
plt.show()

print("Inference: This plot shows the training and validation loss for the optimized TensorFlow/Keras model. A decreasing trend in both indicates effective learning, while a divergence (validation loss increasing) could suggest overfitting.")

Plot saved to /content/drive/MyDrive/output_folder/TF_Optimized_Model_Accuracy_vs_Epoch.pdf
Inference: This plot displays the training and validation accuracy of the optimized TensorFlow/Keras model over epochs. It helps in assessing the model's learning progress and detecting potential overfitting or underfitting after hyperparameter optimization.
Plot saved to /content/drive/MyDrive/output_folder/TF_Optimized_Model_Loss_vs_Epoch.pdf
Inference: This plot shows the training and validation loss for the optimized TensorFlow/Keras model. A decreasing trend in both indicates effective learning, while a divergence (validation loss increasing) could suggest overfitting.


#### 8.4. Best Model Accuracy Comparison Plot (TensorFlow/Keras)

In [55]:
comparison_accuracies_tf = pd.DataFrame({
    'Model': ['Baseline', 'Optimized'],
    'Accuracy': [accuracy_tf, optimized_accuracy_tf]
})

fig_comp_tf = plt.figure(figsize=(8, 5))
sns.barplot(x='Model', y='Accuracy', data=comparison_accuracies_tf, palette=['skyblue', 'lightcoral'])
plt.ylim(0.8, 1.0)
plt.title('Baseline vs. Optimized Model Test Accuracy (TensorFlow/Keras)')
plt.ylabel('Test Accuracy')
plt.grid(axis='y', linestyle='--', alpha=0.7)
save_plot_tf(fig_comp_tf, 'TF_Model_Accuracy_Comparison.pdf')
plt.show()

print("Inference: This bar chart provides a direct visual comparison of the test accuracies between the baseline and optimized TensorFlow/Keras models, clearly indicating the impact of hyperparameter tuning.")

Plot saved to /content/drive/MyDrive/output_folder/TF_Model_Accuracy_Comparison.pdf
Inference: This bar chart provides a direct visual comparison of the test accuracies between the baseline and optimized TensorFlow/Keras models, clearly indicating the impact of hyperparameter tuning.


/tmp/ipykernel_618/1719543227.py:7: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.

  sns.barplot(x='Model', y='Accuracy', data=comparison_accuracies_tf, palette=['skyblue', 'lightcoral'])


### 9. Results (TensorFlow/Keras)

#### Best Hyperparameters (TensorFlow/Keras)

In [56]:
print("| Parameter                 | Value |\n| :------------------------ | :---- |")
print(f"| Hidden Layers             | {best_hps_tf.get('num_hidden_layers')} |")
for i in range(best_hps_tf.get('num_hidden_layers')):
    print(f"| Hidden Neurons Layer {i+1}        | {best_hps_tf.get(f'units_{i}')} |")
    print(f"| Activation Layer {i+1}          | {best_hps_tf.get(f'activation_{i}')} |")
    dropout_enabled = best_hps_tf.get(f'dropout_{i}_enabled')
    if dropout_enabled:
        print(f"| Dropout Rate Layer {i+1}        | {best_hps_tf.get(f'dropout_{i}_rate')} |")
print(f"| Learning Rate             | {best_hps_tf.get('learning_rate')} |")
print(f"| Optimizer                 | {best_hps_tf.get('optimizer')} |")
print(f"| Batch Size                | {best_hps_tf.get('batch_size')} |")

print(f"| Epochs                    | {epochs_optimized_tf} |")
print(f"| Tuner Best Test Accuracy  | {accuracy_best_tf:.4f} |")
print(f"| Optimized Test Accuracy   | {optimized_accuracy_tf:.4f} |")

| Parameter                 | Value |
| :------------------------ | :---- |
| Hidden Layers             | 1 |
| Hidden Neurons Layer 1        | 256 |
| Activation Layer 1          | relu |
| Learning Rate             | 0.001 |
| Optimizer                 | adam |
| Batch Size                | 64 |
| Epochs                    | 20 |
| Tuner Best Test Accuracy  | 0.8770 |
| Optimized Test Accuracy   | 0.8779 |


In [57]:
print("| Metric        | Baseline | Optimized |\n| :------------ | :------- | :-------- |")
print(f"| Accuracy      | {accuracy_tf:.4f}   | {optimized_accuracy_tf:.4f} |")
print(f"| Precision     | {precision_tf:.4f}   | {optimized_precision_tf:.4f} |")
print(f"| Recall        | {recall_tf:.4f}   | {optimized_recall_tf:.4f} |")
print(f"| F1-score      | {f1_tf:.4f}   | {optimized_f1_tf:.4f} |")
print(f"| Training Time | {training_time_tf:.2f}s | {optimized_training_time_tf:.2f}s |")

| Metric        | Baseline | Optimized |
| :------------ | :------- | :-------- |
| Accuracy      | 0.8891   | 0.8779 |
| Precision     | 0.8897   | 0.8813 |
| Recall        | 0.8891   | 0.8779 |
| F1-score      | 0.8891   | 0.8789 |
| Training Time | 198.54s | 132.11s |
